# Multi-task inference + evaluation (vLLM)

Runs a trained model over every scoreable task in this repo with vLLM's grammar-constrained decoding, then scores each one with its own real scorer -- a real-model version of `inference_and_eval.eval.run_eval._self_check_all()` (which does the same loop with gold fed back as the "prediction" instead of a model completion).

This composes two things that already exist rather than reimplementing them:

- **`inference_and_eval/inference/vllm_structured_infer.py`** -- `run(dataset, model, ...)`: loads the dataset's own rendered `{system, user}` prompt (identical to what training saw, gold turn dropped), attaches `multitask.target_schema.json_schema(task)` as a decoding-time grammar so output is on-schema by construction, and returns `{id: completion}`.
- **`multitask/eval/run_eval.py`** -- `run_eval(dataset, completions)`: converts each dataset's completions back to its native format (BRAT `.ann`, n2c2 XML, TSV, ...) where needed and invokes that dataset's *own* scorer -- the official one where it exists, this project's own verified reimplementation otherwise. See `README.md`'s "Per-task evaluation status" table for which is which and what each number's known ceiling is.

**Not runnable in this sandbox** (no `vllm`, no GPU) -- written to be copied into an environment that has both. Nothing in this notebook has been executed here; read it before running it.

## Prerequisites

Use the project venv at `/Users/L069561/workspace/py/venv/` (already has `vllm`, which brings in `xgrammar` for `structured_outputs_config`/`StructuredOutputsParams`) -- do not `pip install` into it or any other environment.

- A model on the HF Hub or a local checkpoint path, with enough GPU memory to load it.
- `perl` on `PATH` (STS's official `correlation-noconfidence.pl`) and a working `python3` (family_history's and cohort_eligibility's official scripts run as subprocesses) -- both already required by this repo's own self-checks, not new for this notebook.
- Run from the repo root, or adjust `sys.path` in the next cell if your working directory differs.

In [ ]:
import sys
from pathlib import Path

# Repo root = two levels up from inference_and_eval/inference/<this file>. Adjust if
# you moved this notebook or are running from a different cwd.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "multitask").exists():
    REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import contextlib
import io
import json
import re

from inference_and_eval.inference.vllm_structured_infer import DEFAULT_MAX_TOKENS, EVAL_DIR, _make_llm
from inference_and_eval.inference.vllm_structured_infer import run as run_inference
from inference_and_eval.eval.run_eval import run_eval

print(f"repo root: {REPO_ROOT}")

## Config

`DATASETS` defaults to every `multitask/data/eval/<name>.jsonl` that actually has rows -- `2022n2c2_track2.jsonl` is stale/empty (superseded by the split `_mimic`/`_uw` datasets, see `README.md`'s Known limitations) and is excluded automatically, not by an explicit skip list.

Start with a small `LIMIT` (e.g. 10-20) to smoke-test the whole pipeline end-to-end quickly -- inference + eval over 22 tasks at full size (the largest, `emrqa-msquad`, alone has 32,739 eval rows) is a long run. Set `LIMIT = None` for the real numbers.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # HF Hub id or local checkpoint path
BACKEND = "xgrammar"                # or "outlines" -- see vllm_structured_infer's module docstring
LIMIT = 20                          # None for a full run; small int for a smoke test

# Sampling. TEMPERATURE=0.0 is greedy/deterministic -- recommended for
# reproducible per-task scoring, and NOT vLLM's own default (1.0). See
# vllm_structured_infer.py's module docstring for why this default was
# chosen deliberately rather than left unset.
# NOTE: RETRIES below only actually re-draw a different completion when
# TEMPERATURE > 0 -- at temperature=0 the seed is irrelevant and a retry
# reproduces the same broken output. If you want the retry mechanism to be
# effective, set TEMPERATURE to a small nonzero value (e.g. 0.1). You do
# NOT need to set a seed in SAMPLING_KWARGS -- the retry loop bumps it
# internally.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = -1
SAMPLING_KWARGS = {}                # any other vllm.SamplingParams field, e.g. {"repetition_penalty": 1.1}

# None (default) never touches the chat template. True/False passes
# chat_template_kwargs={"enable_thinking": ...} to LLM.chat() -- only
# meaningful for a Qwen3-style reasoning-capable model, and silently
# no-op'd with a printed note on a vLLM build whose LLM.chat() doesn't
# accept chat_template_kwargs yet (confirmed absent in 0.6.6.post1,
# present in 0.19.0).
ENABLE_THINKING = None

# Compile the JSON grammar with any_whitespace=False, forbidding whitespace
# between JSON tokens. Fixes the failure mode where grammar-constrained
# decoding loops on tabs/spaces until max_tokens without ever opening '{'
# (legal under the grammar, since inter-token whitespace is valid JSON, but
# useless as output). ENGINE-LEVEL config -- set here, threaded into
# _make_llm, NOT into per-request SamplingParams (vLLM 0.19.0 has the field
# on StructuredOutputsParams but no code path reads it; only
# structured_outputs_config is honored). Needs a vLLM build whose
# structured_outputs_config accepts the key + the xgrammar or guidance
# backend; silently dropped with a printed note on an older vLLM.
DISABLE_ANY_WHITESPACE = True

# How many times to re-draw an obviously-problematic sample (lone surrogate /
# unparseable JSON / schema violation) in isolation before storing what it
# produced. Each retry bumps the sampling seed; only effective with
# TEMPERATURE > 0 (see note above). Truncation is NOT retried (reproduces
# deterministically at the same max_tokens). 0 disables retry.
RETRIES = 2

# vLLM engine (LLM constructor) args -- engine-level, decided once at model
# load below. Pass only the ones to override; vLLM applies its own defaults
# for the rest (tensor_parallel_size=1, gpu_memory_utilization=0.9,
# max_num_seqs=256, max_model_len=auto from model config). Examples:
#   LLM_KWARGS = {"tensor_parallel_size": 2}                 # shard across 2 GPUs
#   LLM_KWARGS = {"gpu_memory_utilization": 0.95}
#   LLM_KWARGS = {"max_model_len": 32768, "max_num_seqs": 64}
# max_model_len: leave unset/None to AUTO-DERIVE from the model's config.json
# (max_position_embeddings) -- you do NOT need to set it to "sync" with the
# model. Set it LOWER only to trade context length for more KV cache / higher
# concurrency. WARNING: it caps prompt + output together, so it MUST exceed
# your longest prompt or vLLM errors on those examples -- this repo's longest
# eval prompts are discharge-me (~14.5k) and 2018_n2c2_track2 (~12.5k), so any
# value below ~15k breaks those datasets.
LLM_KWARGS = {}

OUT_DIR = REPO_ROOT / "multitask" / "inference" / "runs" / "latest"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Every dataset with a non-empty eval file, in the order they appear on disk.
DATASETS = sorted(
    p.stem for p in EVAL_DIR.glob("*.jsonl")
    if p.stat().st_size > 0
)
print(f"{len(DATASETS)} datasets:")
for d in DATASETS:
    print(f"  {d}")

## 1. Load the model once

Loaded a single time and reused across every dataset below -- `vllm_structured_infer.run()` accepts an `llm=` override for exactly this (it otherwise reloads the model on every call, fine for the script's one-dataset-per-process CLI usage, wasteful looped over 22 datasets in one process).

In [ ]:
llm = _make_llm(
    MODEL, BACKEND,
    disable_any_whitespace=DISABLE_ANY_WHITESPACE,
    llm_kwargs=LLM_KWARGS or None,
)
print(
    f"loaded {MODEL} with backend={BACKEND!r}, "
    f"disable_any_whitespace={DISABLE_ANY_WHITESPACE!r}"
    + (f", llm_kwargs={LLM_KWARGS}" if LLM_KWARGS else "")
)

## 2. Run inference for every task

One call to `run()` per dataset: builds prompts from `multitask/data/eval/<dataset>.jsonl`, attaches the per-task decoding grammar, generates, and writes `{id: completion}` to `OUT_DIR/<dataset>.json` (so a crash partway through this loop doesn't lose already-completed datasets -- rerun this cell and it just overwrites, or narrow `DATASETS` to what's left).

`run()` already prints, per dataset, how many completions hit `max_tokens` (`finish_reason == "length"`) and how many parsed as valid JSON but failed `target_schema.validate()` -- read both counts before trusting a low score on that dataset; either one means the *decoding*, not the model's actual task competence, is what's being measured. See `vllm_structured_infer.py`'s module docstring for why `max_tokens` is a correctness parameter here, not just a cost knob.

A dataset that raises is caught and skipped rather than killing the whole run -- `ERRORS` collects what broke.

In [ ]:
all_completions = {}
ERRORS = {}

for dataset in DATASETS:
    print(f"\n=== inference: {dataset} ===")
    try:
        completions = run_inference(
            dataset, MODEL, backend=BACKEND, limit=LIMIT,
            out_path=OUT_DIR / f"{dataset}.json", llm=llm,
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            enable_thinking=ENABLE_THINKING, sampling_kwargs=SAMPLING_KWARGS,
            retries=RETRIES,
        )
        all_completions[dataset] = completions
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        ERRORS[dataset] = f"inference: {type(exc).__name__}: {exc}"

print(f"\ncompleted inference for {len(all_completions)}/{len(DATASETS)} datasets")
if ERRORS:
    print(f"failed: {list(ERRORS)}")

### Optional: reload completions from disk instead of re-running inference

Skip this cell if you just ran the inference loop above in this session. Useful for re-scoring a previous run (e.g. after a `convert_back`/scorer fix) without regenerating.

In [ ]:
# all_completions = {}
# for dataset in DATASETS:
#     path = OUT_DIR / f"{dataset}.json"
#     if path.exists():
#         all_completions[dataset] = json.loads(path.read_text(encoding="utf-8"))
# print(f"reloaded {len(all_completions)}/{len(DATASETS)} datasets from {OUT_DIR}")

## 3. Score every task with its own real scorer

`run_eval(dataset, completions)` dispatches each dataset to its real scorer -- BRAT `.ann` conversion + `brat_scorer` for the three BRAT-family tasks, the official Perl/Python scripts for STS/family_history/cohort_eligibility, and this project's own verified reimplementations for everything else (disorder_normalization, i2b2, discharge-me, qa_span, the simple classification/QA tasks). See `README.md`'s "Per-task evaluation status" table for exactly which is which.

**Deliberately not reduced to one number.** Every task here is scored with a different metric on a different scale (strict/lenient F1, accuracy, Pearson r, ROUGE/BLEU, exact-match+token-F1, ...) -- collapsing them into a single leaderboard column would misrepresent tasks that aren't comparable, which is also why the README never does this. Each dataset's real output (JSON metrics dict, or the official script's own printed table) is captured and printed below, in full, per dataset. Section 4 builds a best-effort convenience summary on top of this -- read it as an index into these blocks, not as the numbers themselves.

In [ ]:
eval_output = {}  # dataset -> captured stdout text from run_eval()

for dataset in DATASETS:
    if dataset not in all_completions:
        continue  # inference failed for this one, see ERRORS above
    print(f"\n=== eval: {dataset} ===")
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            run_eval(dataset, all_completions[dataset])
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        ERRORS[dataset] = f"eval: {type(exc).__name__}: {exc}"
        continue
    text = buf.getvalue()
    eval_output[dataset] = text
    print(text)

print(f"\nscored {len(eval_output)}/{len(DATASETS)} datasets")
if ERRORS:
    print(f"failed at some stage: {list(ERRORS)}")

## 4. Summary

A best-effort *index* into the real output captured above, not a substitute for it -- output formats genuinely differ (JSON metrics dicts, `brat_scorer`'s own `P=... R=... F1=...` print lines, the official Perl/Python scripts' own fixed-width tables), so `_headline()` below tries a few extraction strategies and falls back to `"(see full output above)"` rather than guessing. Always read the full block for a dataset before trusting its headline number, especially anything near 0 or 1.0 -- that's as likely to be a parse/format mismatch as a real result.

In [ ]:
def _headline(text):
    """Best-effort (metric_name, value) from one dataset's captured
    run_eval() stdout, or (None, None) if nothing recognizable was found.
    Tried in order against real output from every scorer family in this
    repo (verified against actual captured text, not just plausible-looking
    regexes -- see the project's own eval-status table for what each
    dataset actually prints): parsed JSON's own headline-shaped keys,
    brat_scorer's combined/overall F1= line (else its first F1= line),
    cohort_eligibility's official "Overall (micro)" row, family_history's
    official "Precision/Prevision Recall F1" header + data row, STS's
    "Pearson" line or bare correlation number."""
    text = text.strip()
    if not text:
        return None, None

    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = None
    if isinstance(obj, dict):
        # depth-first search for the first headline-shaped key
        stack = [obj]
        while stack:
            d = stack.pop()
            for key in ("f1", "accuracy", "pearson", "exact_match", "assertion_accuracy"):
                if key in d and isinstance(d[key], (int, float)):
                    return key, d[key]
            for v in d.values():
                if isinstance(v, dict):
                    stack.append(v)

    lines = text.splitlines()
    for line in lines:
        if ("combined" in line or "overall" in line.lower()) and "F1=" in line:
            m = re.search(r"F1=([\d.]+)", line)
            if m:
                return line.split(":")[0].strip(), float(m.group(1))
    all_f1 = re.findall(r"F1=([\d.]+)", text)
    if all_f1:
        # first, not last: brat_scorer prints its primary axis (tags,
        # triggers, ...) before any secondary one, and datasets with no
        # "combined" line (e.g. 2018_n2c2_track2's tags/relations) have no
        # other signal for which axis is "the" headline number
        return "F1", float(all_f1[0])

    for line in lines:
        if line.strip().startswith("Overall (micro)"):
            nums = re.findall(r"[\d.]+", line)
            if len(nums) >= 2:
                return "Overall (micro) F(b=1)", float(nums[-2])  # last col is AUC, not F(b=1)
    for i, line in enumerate(lines):
        if re.search(r"(Precision|Prevision)\s+Recall\s+F", line, re.I) and i + 1 < len(lines):
            nums = re.findall(r"[\d.]+", lines[i + 1])
            if nums:
                return "F1 (last col)", float(nums[-1])
    m = re.search(r"Pearson.*?([\d.]+)\s*$", text, re.I | re.M)
    if m:
        return "Pearson r", float(m.group(1))
    last_line = lines[-1].strip() if lines else ""
    if re.fullmatch(r"[\d.]+", last_line):
        return "value", float(last_line)

    return None, None


rows = []
for dataset in DATASETS:
    n = len(all_completions.get(dataset, {}))
    status = "error" if dataset in ERRORS else ("ok" if dataset in eval_output else "skipped")
    metric, value = _headline(eval_output.get(dataset, ""))
    rows.append({
        "dataset": dataset,
        "n": n,
        "status": status,
        "headline_metric": metric or "(see full output above)",
        "headline_value": value,
        "error": ERRORS.get(dataset, ""),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

## 5. Optional: offset-free doc-level addon

`multitask/eval/doc_level.py` (via `run_doc_level_eval`) scores the same completions on a document-level, offset-free axis for the 5 tasks it covers (`2018_n2c2_track2`, `2022n2c2_track1`, `2022n2c2_track2_mimic`/`_uw`, `2010_i2b2_va`) instead of the real offset-based scorer. Useful as a second opinion when you suspect the real scorer's number is depressed by `reanchor()` ambiguity rather than genuine model error (see `README.md`'s "Residual text-ambiguity ceiling" note) -- not a replacement for section 3's real numbers.

In [ ]:
from inference_and_eval.eval.run_eval import run_doc_level_eval

DOC_LEVEL_DATASETS = ["2018_n2c2_track2", "2022n2c2_track1", "2022n2c2_track2_mimic", "2022n2c2_track2_uw", "2010_i2b2_va"]

for dataset in DOC_LEVEL_DATASETS:
    if dataset not in all_completions:
        continue
    print(f"\n=== doc-level (offset-free): {dataset} ===")
    try:
        run_doc_level_eval(dataset, all_completions[dataset])
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")

## Known caveats

- **`max_tokens` is a correctness parameter, not a cost knob, for the four JSON-heavy IE tasks.** Under grammar-constrained decoding, hitting the cap mid-object leaves the JSON unclosed -- that fails to parse and scores as a total miss, not a partial one. `DEFAULT_MAX_TOKENS` in `vllm_structured_infer.py` is sized off measured gold-target lengths per task; if section 2's per-dataset output reports truncated completions, raise `--max-tokens`/the relevant entry in `DEFAULT_MAX_TOKENS` before trusting that dataset's score in section 3.
- **The decoding grammar constrains JSON *shape*, not semantic correctness.** Entity/span `"text"` fields are unconstrained strings (the grammar doesn't force them to be substrings of the input note -- that's `convert_back/`'s job, at scoring time, via `reanchor()`), and SHAC's `{"role","name","value"}` cross-field constraint (legal `value`s depend on `name`) isn't expressible in a plain JSON Schema. `target_schema.validate()` catches the second class after generation -- see section 2's printed violation counts.
- **`discharge-me` needs its output in the exact literal header format the instruction states** (`"DISCHARGE INSTRUCTIONS:\n...\n\nBRIEF HOSPITAL COURSE:\n..."`) for `eval/discharge_me.py`'s `parse_sections()` to split the two scored sections apart. A model whose sampled output drifts from that (casing, spacing) silently scores 0 on both sections with no error raised -- if this dataset's numbers look suspiciously low, check the raw completions in `OUT_DIR/discharge-me.json` before assuming it's a model-quality result.
- **Every reported number here is the one this repo's own `README.md` documents, including its known non-bug ceilings** (`reanchor()` ambiguity on the BRAT-family tasks, i2b2's public-vs-challenge split not being comparable to published numbers, `disorder_normalization`'s discontinuous-mention ceiling, ...). Read the "Per-task evaluation status" and "Known limitations" sections there before concluding a low score reflects the model rather than the metric.
- Nothing in this notebook has been executed in this sandbox (no `vllm`, no GPU) -- every function it calls (`run_inference`/`run`, `run_eval`, `run_doc_level_eval`) is exercised elsewhere in this repo's own test/self-check paths, but the notebook's own control flow (the loops, `_headline()`, error handling) has only been unit-tested against representative captured output, not run end-to-end against a real model.